In [ ]:
import pandas as pd
import glob

# Match all the split files (adjust pattern if needed)
files = sorted(glob.glob("Data/vesseldata/vesselpositions_part_*.csv"))

# Read and combine all files
df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

print(df.shape)  # Check rows/columns


(795814, 7)


In [2]:
df["upload-timestamp"] = pd.to_datetime(df["upload-timestamp"], utc=True, errors='coerce')


In [3]:
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
import contextily as ctx
from datetime import datetime, timedelta

In [4]:
# 1️⃣ Convert DataFrame to GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df["lon"], df["lat"])],
    crs="EPSG:4326"
)

In [8]:
def MapOnTime(gdf, end_time):
    
    end_time = end_time.tz_convert('UTC') if end_time.tzinfo else end_time
    start_time = end_time - pd.Timedelta(minutes=3)

    gdf = gdf[
        (gdf["upload-timestamp"] >= start_time) &
        (gdf["upload-timestamp"] <= end_time)
    ]
    
    #speed = 0
    #gdf_speed = gdf[gdf["speed-in-centimeters-per-second"] == speed]
    gdf_speed = gdf
    gdf_speed = gdf_speed.to_crs(epsg=3857)
    fig, ax = plt.subplots(figsize=(10, 10))

    gdf_speed.plot(ax=ax, markersize=0)  # empty plot to set extent
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

    gdf_speed["length_m"] = gdf_speed["radar-length-cm"] / 100

    # 5️⃣ Plot points with Seaborn
    sns.scatterplot(
        x=gdf_speed.geometry.x,
        y=gdf_speed.geometry.y,
        size=gdf_speed["length_m"],
        sizes=(20, 200),
        hue=gdf_speed["id"],  # color by ID if needed
        palette="tab20",
        s=20,
        alpha=0.7,
        legend=False,  # remove legend if you want
        ax=ax
    )

    from pyproj import Transformer

    transformer = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

    xticks = ax.get_xticks()
    yticks = ax.get_yticks()
    
    ax.set_yticks(yticks)
    ax.set_xticks(xticks)

    ax.set_yticklabels([f"{transformer.transform(0, y)[1]:.2f}" for y in yticks])
    ax.set_xticklabels([f"{transformer.transform(0, x)[1]:.2f}" for x in xticks])

    # Suppose start_time and end_time are tz-aware Timestamps
    start_str = start_time.strftime("%b %d, %Y %H:%M")
    end_str = end_time.strftime("%b %d, %Y %H:%M")
    
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.title(f"Vessels > 45m between {start_str} - {end_str}")
    plt.show()



In [9]:
import ipywidgets as widgets
from IPython.display import display


In [10]:
timestamps = pd.to_datetime(gdf["upload-timestamp"].sort_values().unique())

time_slider = widgets.SelectionSlider(
    options=timestamps,
    description='Time:',
    orientation='horizontal',
    layout={'width': '800px'},
    continuous_update=False
)

In [ ]:
def update_map(selected_time):
    MapOnTime(gdf, selected_time)

In [ ]:
widgets.interactive(update_map, selected_time=time_slider)

interactive(children=(SelectionSlider(continuous_update=False, description='Time:', index=330317, layout=Layou…